# Bybit Data Downloader

### Objective:
- Download and save historical price data from the website as CSV files.

In [7]:
# Collecting all imports in one place
import pandas as pd
import requests
import time
import random
import re
import os
import gzip
import shutil

## Function for data downloading

In [4]:
#Function for downloading trade data from Bybit website
def download_file(market_type='spot',      #Options: spot, trading (futures)
                  time_aggregated='daily', #For spot: daily, monthly. For trading: daily. 
                  date='2024-01-01',       #Format for monthly: 2020-08, for daily: 2020-08-08
                  ticker='BTCUSDT'):
    
    
    if time_aggregated == 'daily' and market_type == 'spot':
        symbol = '_'
    if time_aggregated == 'daily' and market_type == 'trading':    
        symbol = ''
    else:
        symbol = '-'
        
    url = f"https://public.bybit.com/{market_type}/{ticker}/{ticker}{symbol}{date}.csv.gz"
    print(url)
    
    #Directory for saving the file
    directory = f"/Users/danielschollenberg/Documents/Трейдинг/Данные/bybit/{market_type}/trades/{ticker}"
    #Create directory if it does not exist
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Directory {directory} created.")
            
    #Bypassing potential blocking:
    #Add random delay from 0 to 2 seconds before request
    delay = random.uniform(0, 2)
    print(f"Delay before request: {delay:.2f} seconds")
    time.sleep(delay)
    #List of User-Agent headers
    user_agents = ['Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
                   'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0.1 Safari/605.1.15',
                   'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:89.0) Gecko/20100101 Firefox/89.0',
                   'Mozilla/5.0 (iPhone; CPU iPhone OS 14_6 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.1 Mobile/15E148 Safari/604.1',
                   'Mozilla/5.0 (Linux; Android 10; SM-G975F) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/89.0.4389.105 Mobile Safari/537.36']
    #Select random User-Agent header
    headers = {'User-Agent': random.choice(user_agents)}
    
    #Create request to get data in streaming mode
    response = requests.get(url, headers=headers, stream=True)
    #If connection successful (status 200) download and save file
    if response.status_code == 200:
        filename = directory + f"/{ticker}_{date}.csv.gz"
        with open(filename, 'wb') as f:
            #Write all content directly to file
            f.write(response.content)  
        print(f"File {filename} successfully downloaded and saved")
    else:
        print(f"Error downloading file: status {response.status_code}, URL: {url}")
        
    #Function for unpacking gz file
    def unpack_gz(gz_path, output_filename):
        with gzip.open(gz_path, 'rb') as f_in:
            with open(output_filename, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        #Delete gz file after extraction
        os.remove(gz_path)
        print(f"Archive {gz_path} deleted after extraction.")
    
    try:
        #Unpack gz file
        unpack_gz(filename, os.path.join(directory, f"{ticker}_{date}.csv"))
        print(f"File {filename} successfully extracted.")
    except:
        print('Error during extraction')

In [49]:
# Example of usage
download_file(market_type='trading', time_aggregated='daily', date='2024-01-01')

https://public.bybit.com/trading/BTCUSDT/BTCUSDT2024-01-01.csv.gz
Файл /Users/danielschollenberg/Documents/Трейдинг/Данные/bybit/trading/trades/BTCUSDT/BTCUSDT_2024-01-01.csv.gz успешно скачан, сохранен на диск
Архив /Users/danielschollenberg/Documents/Трейдинг/Данные/bybit/trading/trades/BTCUSDT/BTCUSDT_2024-01-01.csv.gz удален после распаковки.
Файл /Users/danielschollenberg/Documents/Трейдинг/Данные/bybit/trading/trades/BTCUSDT/BTCUSDT_2024-01-01.csv.gz успешно распакован.
